# 深度學習性能分析工具完全指南
:label:`sec_profiling_tools`

**「不能測量，就無法優化」** - 這是性能優化的黃金法則。

本章將深入介紹深度學習中最重要的性能分析工具，幫助你：

- 🔍 **精準定位性能瓶頸**：CPU、GPU、記憶體、I/O
- ⏱️ **測量真實性能**：避免被表面數字誤導
- 📊 **視覺化分析結果**：使用 TensorBoard、Chrome Trace
- 🎯 **AI 輔助優化建議**：讓 AI 幫你分析 profiling 結果

## 目錄
1. [PyTorch Profiler 完全指南](#1-pytorch-profiler-完全指南)
2. [TensorBoard 集成](#2-tensorboard-集成)
3. [CUDA 事件計時](#3-cuda-事件計時)
4. [記憶體分析](#4-記憶體分析)
5. [NVIDIA 工具鏈](#5-nvidia-工具鏈)
6. [性能瓶頸診斷案例](#6-性能瓶頸診斷案例)
7. [AI 輔助性能分析](#7-ai-輔助性能分析)

In [ ]:
# 導入必要的庫
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.profiler import profile, record_function, ProfilerActivity
from torch.utils.tensorboard import SummaryWriter
import time
import numpy as np
from torchvision import models
import json
from pathlib import Path

# 檢查環境
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch 版本: {torch.__version__}")
print(f"使用設備: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA 版本: {torch.version.cuda}")

## 1. PyTorch Profiler 完全指南

### 1.1 基本使用

PyTorch Profiler 是官方提供的性能分析工具，可以追蹤：
- CPU 時間
- CUDA 時間
- CUDA 記憶體使用
- 操作符調用堆棧

In [ ]:
# 創建一個簡單的模型用於演示
class DemoModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 3, padding=1)
        self.conv2 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2)
        self.fc = nn.Linear(128 * 8 * 8, 10)
    
    def forward(self, x):
        # 使用 record_function 標記代碼段
        with record_function("CONV_BLOCK_1"):
            x = F.relu(self.conv1(x))
            x = self.pool(x)
        
        with record_function("CONV_BLOCK_2"):
            x = F.relu(self.conv2(x))
            x = self.pool(x)
        
        with record_function("FC_BLOCK"):
            x = x.view(x.size(0), -1)
            x = self.fc(x)
        
        return x

# 基本 profiling
def basic_profiling_example():
    model = DemoModel().to(device)
    input_data = torch.randn(32, 3, 32, 32, device=device)
    
    print("\n" + "="*70)
    print("基本 Profiling 示例")
    print("="*70)
    
    # 設置 profiler
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)
    
    with profile(
        activities=activities,
        record_shapes=True,  # 記錄張量形狀
        profile_memory=True,  # 記錄記憶體使用
        with_stack=True  # 記錄調用堆棧
    ) as prof:
        with record_function("model_inference"):
            output = model(input_data)
    
    # 打印結果（按 CUDA 時間排序）
    print("\n按 CUDA 時間排序的前 10 個操作：")
    print(prof.key_averages().table(
        sort_by="cuda_time_total" if torch.cuda.is_available() else "cpu_time_total",
        row_limit=10
    ))
    
    # 按自定義標記分組
    print("\n按自定義標記分組：")
    print(prof.key_averages(group_by_input_shape=False).table(
        sort_by="cuda_time_total" if torch.cuda.is_available() else "cpu_time_total",
        row_limit=10
    ))
    
    return prof

prof = basic_profiling_example()

### 1.2 導出 Chrome Trace

Chrome Trace 提供時間線視圖，可以在瀏覽器中查看：

In [ ]:
# 導出為 Chrome Trace 格式
output_dir = Path("./profiler_outputs")
output_dir.mkdir(exist_ok=True)

trace_file = output_dir / "trace.json"
prof.export_chrome_trace(str(trace_file))

print(f"\n✓ Chrome Trace 已導出到: {trace_file}")
print("\n如何查看:")
print("  1. 打開 Chrome 瀏覽器")
print("  2. 訪問 chrome://tracing")
print(f"  3. 點擊 'Load' 並選擇 {trace_file}")
print("\n提示: 使用 W/A/S/D 鍵導航時間線")

### 1.3 高級 Profiling：訓練循環

In [ ]:
def profile_training_loop():
    """分析完整的訓練循環"""
    print("\n" + "="*70)
    print("訓練循環 Profiling")
    print("="*70)
    
    model = DemoModel().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()
    
    # 生成假數據
    data = torch.randn(32, 3, 32, 32, device=device)
    target = torch.randint(0, 10, (32,), device=device)
    
    activities = [ProfilerActivity.CPU]
    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)
    
    # 使用 schedule 控制 profiling 時機
    with profile(
        activities=activities,
        schedule=torch.profiler.schedule(
            wait=1,      # 跳過前 1 個 step
            warmup=1,    # 預熱 1 個 step
            active=3,    # 記錄 3 個 step
            repeat=2     # 重複 2 次
        ),
        on_trace_ready=torch.profiler.tensorboard_trace_handler('./profiler_logs'),
        record_shapes=True,
        profile_memory=True,
        with_stack=True
    ) as prof:
        for step in range(10):
            with record_function(f"## Step {step}"):
                # 前向傳播
                with record_function("forward"):
                    output = model(data)
                    loss = criterion(output, target)
                
                # 反向傳播
                with record_function("backward"):
                    optimizer.zero_grad()
                    loss.backward()
                
                # 優化器步驟
                with record_function("optimizer_step"):
                    optimizer.step()
            
            # 通知 profiler 進入下一步
            prof.step()
            
            if step % 2 == 0:
                print(f"  Step {step} 完成")
    
    print("\n✓ TensorBoard 日誌已保存到 ./profiler_logs")
    print("\n查看方式:")
    print("  tensorboard --logdir=./profiler_logs")
    print("  然後訪問 http://localhost:6006")
    
    return prof

training_prof = profile_training_loop()

## 2. TensorBoard 集成

### 2.1 實時性能監控

In [ ]:
def tensorboard_profiling_demo():
    """使用 TensorBoard 進行實時性能監控"""
    writer = SummaryWriter('runs/profiling_demo')
    
    model = models.resnet18().to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
    
    print("\n" + "="*70)
    print("TensorBoard Profiling 演示")
    print("="*70)
    
    # 記錄模型圖
    dummy_input = torch.randn(1, 3, 224, 224, device=device)
    writer.add_graph(model, dummy_input)
    
    # 訓練並記錄性能指標
    for epoch in range(3):
        epoch_start = time.time()
        
        for step in range(10):
            step_start = time.time()
            
            # 生成假數據
            data = torch.randn(32, 3, 224, 224, device=device)
            target = torch.randint(0, 1000, (32,), device=device)
            
            # 訓練步驟
            optimizer.zero_grad()
            output = model(data)
            loss = F.cross_entropy(output, target)
            loss.backward()
            optimizer.step()
            
            step_time = time.time() - step_start
            
            # 記錄到 TensorBoard
            global_step = epoch * 10 + step
            writer.add_scalar('Loss/train', loss.item(), global_step)
            writer.add_scalar('Time/step', step_time, global_step)
            writer.add_scalar('Throughput/images_per_sec', 32/step_time, global_step)
            
            if torch.cuda.is_available():
                writer.add_scalar('GPU/memory_allocated_MB', 
                                torch.cuda.memory_allocated()/1024**2, global_step)
        
        epoch_time = time.time() - epoch_start
        print(f"  Epoch {epoch}: {epoch_time:.2f}s")
    
    writer.close()
    print("\n✓ TensorBoard 日誌已保存")
    print("  運行: tensorboard --logdir=runs")

tensorboard_profiling_demo()

## 3. CUDA 事件計時

### 3.1 精確的 GPU 計時

In [ ]:
class CUDATimer:
    """精確的 CUDA 計時器"""
    def __init__(self):
        self.start_event = torch.cuda.Event(enable_timing=True)
        self.end_event = torch.cuda.Event(enable_timing=True)
    
    def start(self):
        self.start_event.record()
    
    def stop(self):
        self.end_event.record()
        torch.cuda.synchronize()
        return self.start_event.elapsed_time(self.end_event)  # ms

if torch.cuda.is_available():
    print("\n=== CUDA 事件計時演示 ===")
    
    model = nn.Linear(1000, 1000).to(device)
    x = torch.randn(100, 1000, device=device)
    
    # 使用 CUDA Timer
    timer = CUDATimer()
    
    times = []
    for _ in range(100):
        timer.start()
        _ = model(x)
        elapsed = timer.stop()
        times.append(elapsed)
    
    print(f"\n平均時間: {np.mean(times):.3f} ms")
    print(f"標準差: {np.std(times):.3f} ms")
    print(f"最小值: {np.min(times):.3f} ms")
    print(f"最大值: {np.max(times):.3f} ms")
    print(f"\n提示: CUDA 事件提供比 time.time() 更精確的 GPU 計時")
else:
    print("需要 CUDA GPU 來演示 CUDA 事件計時")

## 4. 記憶體分析

### 4.1 GPU 記憶體追蹤

In [ ]:
def memory_profiling_demo():
    """詳細的記憶體分析"""
    if not torch.cuda.is_available():
        print("需要 CUDA GPU 來進行記憶體分析")
        return
    
    print("\n" + "="*70)
    print("GPU 記憶體分析")
    print("="*70)
    
    # 重置記憶體統計
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    
    def print_memory_stats(stage):
        """打印當前記憶體狀態"""
        allocated = torch.cuda.memory_allocated() / 1024**2
        reserved = torch.cuda.memory_reserved() / 1024**2
        max_allocated = torch.cuda.max_memory_allocated() / 1024**2
        
        print(f"\n{stage}:")
        print(f"  已分配: {allocated:.2f} MB")
        print(f"  已預留: {reserved:.2f} MB")
        print(f"  峰值: {max_allocated:.2f} MB")
    
    print_memory_stats("初始狀態")
    
    # 創建模型
    model = models.resnet50().to(device)
    print_memory_stats("加載模型後")
    
    # 前向傳播
    x = torch.randn(32, 3, 224, 224, device=device)
    output = model(x)
    print_memory_stats("前向傳播後")
    
    # 反向傳播
    loss = output.sum()
    loss.backward()
    print_memory_stats("反向傳播後")
    
    # 使用 memory_snapshot 獲取詳細信息
    print("\n" + "="*70)
    print("記憶體快照分析")
    print("="*70)
    
    snapshot = torch.cuda.memory_snapshot()
    # 保存快照供離線分析
    snapshot_file = output_dir / "memory_snapshot.pickle"
    torch.cuda._memory_viz.trace_plot(snapshot, str(output_dir / "memory_plot.html"))
    
    print(f"\n✓ 記憶體可視化已保存到 {output_dir / 'memory_plot.html'}")
    print("  在瀏覽器中打開該文件查看詳細的記憶體使用情況")

memory_profiling_demo()

### 4.2 記憶體洩漏檢測

In [ ]:
def detect_memory_leak():
    """檢測記憶體洩漏"""
    if not torch.cuda.is_available():
        print("需要 CUDA GPU 來檢測記憶體洩漏")
        return
    
    print("\n=== 記憶體洩漏檢測 ===")
    
    model = nn.Linear(1000, 1000).to(device)
    
    memory_history = []
    
    for i in range(10):
        x = torch.randn(100, 1000, device=device)
        output = model(x)
        loss = output.sum()
        loss.backward()
        
        # 記錄記憶體使用
        mem = torch.cuda.memory_allocated() / 1024**2
        memory_history.append(mem)
        
        if i % 2 == 0:
            print(f"  Iteration {i}: {mem:.2f} MB")
    
    # 分析趨勢
    growth = memory_history[-1] - memory_history[0]
    print(f"\n記憶體增長: {growth:.2f} MB")
    
    if growth > 10:  # 閾值
        print("⚠️  警告: 檢測到可能的記憶體洩漏！")
        print("\n常見原因:")
        print("  1. 沒有調用 optimizer.zero_grad()")
        print("  2. 保存了不必要的中間張量")
        print("  3. 在循環中累積梯度")
    else:
        print("✓ 未檢測到明顯的記憶體洩漏")

detect_memory_leak()

## 5. NVIDIA 工具鏈

### 5.1 nvidia-smi 監控

In [ ]:
print("""
=== NVIDIA 工具使用指南 ===

1. nvidia-smi (實時監控)
   # 基本信息
   nvidia-smi
   
   # 持續監控（每秒刷新）
   nvidia-smi -l 1
   
   # 只顯示記憶體使用
   nvidia-smi --query-gpu=memory.used,memory.total --format=csv

2. nvtop (互動式監控)
   # 安裝: sudo apt install nvtop
   nvtop

3. gpustat (Python 包)
   # 安裝: pip install gpustat
   gpustat -i 1  # 每秒刷新

4. NVIDIA Nsight Systems (詳細分析)
   # 記錄 profiling 數據
   nsys profile -o output python script.py
   
   # 使用 GUI 查看
   nsight-sys output.nsys-rep

5. NVIDIA Nsight Compute (Kernel 分析)
   # 分析特定 kernel
   ncu --set full -o output python script.py
   
   # 查看結果
   ncu-ui output.ncu-rep
""")

## 6. 性能瓶頸診斷案例

### 6.1 案例 1: 數據加載瓶頸

In [ ]:
def diagnose_dataloader_bottleneck():
    """診斷數據加載瓶頸"""
    print("\n" + "="*70)
    print("案例 1: 數據加載瓶頸診斷")
    print("="*70)
    
    from torch.utils.data import TensorDataset, DataLoader
    
    # 創建假數據集
    dataset = TensorDataset(
        torch.randn(1000, 3, 224, 224),
        torch.randint(0, 10, (1000,))
    )
    
    model = models.resnet18().to(device)
    model.eval()
    
    # 測試不同的 num_workers
    num_workers_list = [0, 2, 4]
    
    results = {}
    
    for num_workers in num_workers_list:
        dataloader = DataLoader(
            dataset, 
            batch_size=32, 
            num_workers=num_workers,
            pin_memory=torch.cuda.is_available()
        )
        
        start = time.time()
        
        with torch.no_grad():
            for data, target in dataloader:
                data = data.to(device, non_blocking=True)
                _ = model(data)
        
        elapsed = time.time() - start
        results[num_workers] = elapsed
        
        print(f"  num_workers={num_workers}: {elapsed:.2f}s")
    
    # 分析結果
    best_workers = min(results, key=results.get)
    print(f"\n✓ 最佳 num_workers: {best_workers}")
    print(f"  加速比: {results[0] / results[best_workers]:.2f}x")

diagnose_dataloader_bottleneck()

### 6.2 案例 2: GPU 利用率低

In [ ]:
def diagnose_low_gpu_utilization():
    """診斷 GPU 利用率低的問題"""
    print("\n" + "="*70)
    print("案例 2: GPU 利用率低診斷")
    print("="*70)
    
    if not torch.cuda.is_available():
        print("需要 CUDA GPU")
        return
    
    model = models.resnet50().to(device)
    
    # 問題 1: 批次太小
    print("\n測試不同批次大小:")
    batch_sizes = [1, 16, 64, 128]
    
    for bs in batch_sizes:
        x = torch.randn(bs, 3, 224, 224, device=device)
        
        # 預熱
        for _ in range(5):
            _ = model(x)
        
        torch.cuda.synchronize()
        start = time.time()
        
        for _ in range(20):
            _ = model(x)
        
        torch.cuda.synchronize()
        elapsed = time.time() - start
        
        throughput = (bs * 20) / elapsed
        print(f"  Batch size {bs:3d}: {throughput:6.1f} images/sec")
    
    print("\n診斷建議:")
    print("  ✓ 增加批次大小可以提高 GPU 利用率")
    print("  ✓ 使用梯度累積如果記憶體不足")
    print("  ✓ 檢查數據加載是否成為瓶頸")

diagnose_low_gpu_utilization()

## 7. AI 輔助性能分析

### 7.1 使用 AI 分析 Profiling 結果

In [ ]:
def generate_ai_optimization_prompt(prof_summary):
    """
    生成給 AI (如 ChatGPT, Claude) 的優化建議提示詞
    """
    prompt = f"""
我正在優化一個深度學習模型，以下是 PyTorch Profiler 的結果摘要：

{prof_summary}

請幫我分析：
1. 主要的性能瓶頸在哪裡？
2. 哪些操作佔用了最多時間？
3. 有哪些具體的優化建議？
4. 建議的優化優先級是什麼？

請提供實用的、可以立即實施的建議。
    """
    return prompt

# 示例使用
print("\n" + "="*70)
print("AI 輔助性能分析")
print("="*70)

# 獲取 profiling 摘要
prof_summary = prof.key_averages().table(
    sort_by="cuda_time_total" if torch.cuda.is_available() else "cpu_time_total",
    row_limit=10
)

# 生成 AI 提示詞
ai_prompt = generate_ai_optimization_prompt(prof_summary)

print("\n將以下提示詞複製給 AI 助手（ChatGPT/Claude）：")
print("=" * 70)
print(ai_prompt)
print("=" * 70)

# 保存到文件
with open(output_dir / "ai_optimization_prompt.txt", "w", encoding="utf-8") as f:
    f.write(ai_prompt)

print(f"\n✓ AI 提示詞已保存到 {output_dir / 'ai_optimization_prompt.txt'}")

## 總結

### 性能分析最佳實踐

1. **系統性分析**：
   - 先用 Profiler 找出瓶頸
   - 然後針對性優化
   - 最後驗證優化效果

2. **工具選擇**：
   - 快速診斷：`time.time()`, `nvidia-smi`
   - 詳細分析：PyTorch Profiler
   - 深度分析：Nsight Systems/Compute

3. **常見瓶頸**：
   - 數據加載慢 → 增加 `num_workers`
   - GPU 利用率低 → 增大批次或使用梯度累積
   - 記憶體不足 → 梯度檢查點、混合精度

4. **優化流程**：
   ```python
   1. Profile → 找出瓶頸
   2. Optimize → 實施優化
   3. Measure → 測量效果
   4. Iterate → 重複直到滿意
   ```

### 進階學習

- [PyTorch Profiler 教程](https://pytorch.org/tutorials/recipes/recipes/profiler_recipe.html)
- [NVIDIA 性能優化指南](https://docs.nvidia.com/deeplearning/performance/)
- [TensorBoard Profiler 插件](https://pytorch.org/tutorials/intermediate/tensorboard_profiler_tutorial.html)